In [5]:
import pandas as pd

# We're combining train + test because, as we found during inspection,
# this dataset isn't a standard train/test split with identical structure —
# 'test' is missing key outcome columns (order_status, delivered_timestamp,
# estimated_delivery_date). We'll confirm and handle this properly below.

train_path = '../data/raw/train/'
test_path = '../data/raw/test/'

# Load train files
customers_train = pd.read_csv(train_path + 'df_Customers.csv')
orders_train = pd.read_csv(train_path + 'df_Orders.csv')
order_items_train = pd.read_csv(train_path + 'df_OrderItems.csv')
payments_train = pd.read_csv(train_path + 'df_Payments.csv')
products_train = pd.read_csv(train_path + 'df_Products.csv')

# Load test files
customers_test = pd.read_csv(test_path + 'df_Customers.csv')
orders_test = pd.read_csv(test_path + 'df_Orders.csv')
order_items_test = pd.read_csv(test_path + 'df_OrderItems.csv')
payments_test = pd.read_csv(test_path + 'df_Payments.csv')
products_test = pd.read_csv(test_path + 'df_Products.csv')

print('Train orders shape:', orders_train.shape)
print('Test orders shape:', orders_test.shape)
print('Test orders columns:', orders_test.columns.tolist())

Train orders shape: (89316, 7)
Test orders shape: (38279, 4)
Test orders columns: ['order_id', 'customer_id', 'order_purchase_timestamp', 'order_approved_at']


In [6]:
print('Customers test columns:', customers_test.columns.tolist())
print('OrderItems test columns:', order_items_test.columns.tolist())
print('Payments test columns:', payments_test.columns.tolist())
print('Products test columns:', products_test.columns.tolist())

Customers test columns: ['customer_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
OrderItems test columns: ['order_id', 'product_id', 'seller_id', 'price', 'shipping_charges']
Payments test columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Products test columns: ['product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [7]:
# attributes repeat once per order and are identical across those
# duplicate rows. This is safe to dedupe without losing any information.
products_train_unique = products_train.drop_duplicates(subset='product_id', keep='first')

# I'm using train only here, not test, because I confirmed test is
# missing order_status, order_delivered_timestamp, and
# order_estimated_delivery_date. Those are exactly the fields I need
# for delivery analysis, so combining them would just introduce a
# large block of nulls into my main dataset.
master = orders_train.merge(order_items_train, on='order_id', how='left')
master = master.merge(payments_train, on='order_id', how='left')
master = master.merge(customers_train, on='customer_id', how='left')
master = master.merge(products_train_unique, on='product_id', how='left')

print('Master table shape:', master.shape)
print('Columns:', list(master.columns))

Master table shape: (89316, 23)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_timestamp', 'order_estimated_delivery_date', 'product_id', 'seller_id', 'price', 'shipping_charges', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [8]:
print(master.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
product_id                          0
seller_id                           0
price                               0
shipping_charges                    0
payment_sequential                  0
payment_type                        0
payment_installments                0
payment_value                       0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
product_category_name             308
product_weight_g                   15
product_length_cm                  15
product_height_cm                  15
product_width_cm                   15
dtype: int64


In [9]:
# 308 rows are missing product_category_name.
# Dropping these rows would also throw away valid price, order, and 
# customer data just because one descriptive field is blank, so instead
# these get labeled "unknown" — keeps them usable in category-level 
# groupings instead of vanishing from the analysis.
master['product_category_name'] = master['product_category_name'].fillna('unknown')

# quick check this worked
print(master['product_category_name'].isnull().sum())
print(master['product_category_name'].value_counts().tail())

0
product_category_name
fashio_female_clothing               2
furniture_mattress_and_upholstery    1
home_comfort_2                       1
security_and_services                1
diapers_and_hygiene                  1
Name: count, dtype: int64


In [10]:
# tail() shows the smallest categories, not "unknown" specifically since
# 308 rows puts it near the top of the distribution, not the bottom.
# checking directly instead.
print(master['product_category_name'].value_counts()['unknown'])

308


In [11]:
# weight, length, height, and width are each missing 15 values.
# checking whether it's the same 15 rows across all four columns, or
# different rows for each — that changes how I handle it.
dimension_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
missing_dims = master[master[dimension_cols].isnull().any(axis=1)]
print('Rows with at least one missing dimension:', len(missing_dims))
print(missing_dims[dimension_cols].isnull().sum())

Rows with at least one missing dimension: 15
product_weight_g     15
product_length_cm    15
product_height_cm    15
product_width_cm     15
dtype: int64


In [12]:
# same 15 rows are missing all four dimension fields together, so this
# looks like incomplete product listings rather than a random glitch.
# 15 out of 89,316 rows (0.02%) is negligible in most cases.
for col in dimension_cols:
    median_val = master[col].median()
    master[col] = master[col].fillna(median_val)

print(master[dimension_cols].isnull().sum())

product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [13]:
# order_approved_at is missing for 9 orders, and order_delivered_timestamp
# is missing for 1,889 orders. Earlier inspection showed these aren't
# random gaps — they correspond to orders that haven't been approved yet
# or haven't been delivered yet (shipped, processing, canceled, etc. all
# logically lack a delivery timestamp). Filling these in with a fake date
# would misrepresent real order states, so they're left as actual nulls.
# Any delivery-time calculations downstream will naturally return null
# for these rows, which is the correct behavior.

# also flagging the 6 "delivered" rows missing a timestamp and the 5
# "canceled" rows that unexpectedly have one — both already documented
# in the data dictionary as known edge cases, not treated as errors here.

print('order_approved_at nulls remaining:', master['order_approved_at'].isnull().sum())
print('order_delivered_timestamp nulls remaining:', master['order_delivered_timestamp'].isnull().sum())

order_approved_at nulls remaining: 9
order_delivered_timestamp nulls remaining: 1889


In [14]:
# all four timestamp columns are currently just plain strings (object/str
# dtype), which means no date math, sorting, or time-based filtering will
# work correctly on them yet. converting them to actual datetime type now.
timestamp_cols = ['order_purchase_timestamp', 'order_approved_at', 
                   'order_delivered_timestamp', 'order_estimated_delivery_date']

for col in timestamp_cols:
    master[col] = pd.to_datetime(master[col])

# confirming the conversion worked
print(master[timestamp_cols].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_timestamp        datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [15]:
# one last full check before exporting — confirming no unexpected nulls
# remain anywhere except the two timestamp columns I intentionally left
# as real nulls (order_approved_at, order_delivered_timestamp).
print(master.isnull().sum())
print()
print(master.shape)


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
product_id                          0
seller_id                           0
price                               0
shipping_charges                    0
payment_sequential                  0
payment_type                        0
payment_installments                0
payment_value                       0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
product_category_name               0
product_weight_g                    0
product_length_cm                   0
product_height_cm                   0
product_width_cm                    0
dtype: int64

(89316, 23)


In [16]:
# saving the cleaned, merged dataset to data/processed/ so it can be
# used directly for SQL loading, EDA, and Power BI without needing to
# redo all this merging and cleaning every time.
master.to_csv('../data/processed/master_cleaned.csv', index=False)

print('Saved to ../data/processed/master_cleaned.csv')
print('Shape:', master.shape)

Saved to ../data/processed/master_cleaned.csv
Shape: (89316, 23)
